Final XGBoost (v2 history features, stable settings), trained on Kaggle and logged to DagsHub.

Features: the 16 tabular features plus the 33 account-history / graph features (the v2 set WITHOUT account age, which the
ablations showed is a calendar artefact with no signal in val/test). Settings: depth 4, min_child_weight 10, lambda 10,
subsample/colsample 0.7, sqrt class weight, early stopping on validation PR-AUC (found in notebooks 03c/03d). Three seeds.

What gets stored on DagsHub (experiment "papertrail"): one run per seed (xgb_v2_final_seed0/1/2) with params, metrics and the model
file (.json); and a summary run (xgb_v2_final) with mean +- std over seeds, a 3-seed ensemble, the feature importances and the
scores file (xgb_scores.parquet) that the blend notebook reads. The notebook also regenerates age_distribution.csv.

How to run: CPU, Internet ON, DAGSHUB_TOKEN secret attached, input papertrail-processed. Save Version -> Save & Run All (Commit).
Expect roughly 35 to 60 minutes. Set QUICK = True in cell 2 for a one-seed look (about 12 minutes).

Reference (03d, v2 history with age, 3 seeds): test PR-AUC 0.442 +- 0.004, F1 0.499, recall@1%FPR 0.675, precision@1000 0.461.



In [ ]:
!pip install -q git+https://github.com/abhibarnwal707/PaperTrail.git
import os
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
import dagshub, mlflow
dagshub.init(repo_owner="abhibarnwal707", repo_name="papertrail", mlflow=True)


In [ ]:
import time as _t; print("[cell 2: configuration] started", _t.strftime("%H:%M:%S"))
import glob
import json
import os

import numpy as np
import polars as pl
import xgboost as xgb

from papertrail.features import add_out_count_24h, build_features, fit_categories
from papertrail.graph_features import HISTORY_FEATURES, add_history_features
from papertrail.metrics import best_f1_threshold, evaluate
from papertrail.split import assert_valid_split, split_summary, temporal_split

QUICK = False                                    # True: one seed
SEEDS = (0,) if QUICK else (0, 1, 2)
SUFFIX = "_quick" if QUICK else ""
FEATURES_HISTORY = [c for c in HISTORY_FEATURES if not c.endswith("age_s")]     # drop the calendar artefact (s_age_s, d_age_s)
print("seeds:", SEEDS, "| history features:", len(FEATURES_HISTORY), "of", len(HISTORY_FEATURES))


In [ ]:
import time as _t; print("[cell 3: features] started", _t.strftime("%H:%M:%S"))
DATA = "/kaggle/input/datasets/abhibarnwal707/papertrail-processed/transactions.parquet"
if not os.path.exists(DATA):
    DATA = glob.glob("/kaggle/input/**/transactions.parquet", recursive=True)[0]
df = add_out_count_24h(pl.read_parquet(DATA).sort(["ts", "txn_id"]))
summary = split_summary(df)
print(summary)
assert_valid_split(summary)

t0 = _t.time()
dfx = add_history_features(df)
print(f"history features in {_t.time() - t0:.0f}s ->", dfx.shape)
train, val, test = temporal_split(dfx)
cats = fit_categories(train)


def to_matrix(part):
    return pl.concat([build_features(part, cats), part.select(FEATURES_HISTORY)], how="horizontal").cast(pl.Float32)


Xtr_df, Xva_df, Xte_df = to_matrix(train), to_matrix(val), to_matrix(test)
feat_names = Xtr_df.columns
X_train, X_val, X_test = Xtr_df.to_numpy(), Xva_df.to_numpy(), Xte_df.to_numpy()
y_train, y_val, y_test = train["label"].to_numpy(), val["label"].to_numpy(), test["label"].to_numpy()
test_ach = (test["pay_format"] == "ACH").to_numpy()
spw = float((y_train == 0).sum() / y_train.sum())
print(X_train.shape, "scale_pos_weight (full) =", round(spw, 1), "-> used: sqrt =", round(spw ** 0.5, 1))


In [ ]:
import time as _t; print("[cell 4: age diagnostic] started", _t.strftime("%H:%M:%S"))
pl.Config.set_tbl_cols(-1)


def split_stats(name, part, label):
    sub = part.filter(pl.col("label") == label)
    new_s = (pl.col("s_tot_out").cast(pl.Int64) + pl.col("s_tot_in").cast(pl.Int64)) == 0
    new_d = (pl.col("d_tot_out").cast(pl.Int64) + pl.col("d_tot_in").cast(pl.Int64)) == 0
    return sub.select(
        pl.lit(name).alias("split"), pl.lit(label).alias("label"), pl.len().alias("n"),
        (pl.col("s_age_s").median() / 86400).alias("med_s_age_days"),
        (pl.col("s_age_s").quantile(0.9) / 86400).alias("p90_s_age_days"),
        (pl.col("d_age_s").median() / 86400).alias("med_d_age_days"),
        new_s.mean().alias("share_sender_new"), new_d.mean().alias("share_receiver_new"),
        (pl.col("s_gap_s").median() / 60).alias("med_s_gap_min"), pl.col("s_gap_s").is_null().mean().alias("s_gap_null"),
    )


age_tbl = pl.concat([split_stats(n, p, lab) for n, p in (("train", train), ("val", val), ("test", test)) for lab in (0, 1)])
print(age_tbl)
age_tbl.write_csv("/kaggle/working/age_distribution.csv")


In [ ]:
import time as _t; print("[cell 5: train] started", _t.strftime("%H:%M:%S"))


def evaluate_scores(p_val, p_test):
    """Same protocol as every other model: threshold from validation, test scored once, plus the ACH-only slice."""
    thr = best_f1_threshold(y_val, p_val)
    res = {"threshold": float(thr)}
    for k, v in evaluate(y_val, p_val, thr).items():
        res[f"val_{k}"] = v
    for k, v in evaluate(y_test, p_test, thr).items():
        res[f"test_{k}"] = v
    for k, v in evaluate(y_test[test_ach], p_test[test_ach], thr).items():
        res[f"test_ach_{k}"] = v
    return res


results, scores, importances = {}, {}, {}
for seed in SEEDS:
    cfg = dict(tree_method="hist", n_estimators=1000, learning_rate=0.05, max_depth=4, min_child_weight=10, reg_lambda=10.0,
               subsample=0.7, colsample_bytree=0.7, scale_pos_weight=float(np.sqrt(spw)), eval_metric="aucpr",
               early_stopping_rounds=50, n_jobs=-1, random_state=seed)
    t0 = _t.time()
    with mlflow.start_run(run_name=f"xgb_v2_final_seed{seed}{SUFFIX}"):
        mlflow.log_params({k: v for k, v in cfg.items() if k != "eval_metric"})
        mlflow.log_param("n_features", len(feat_names))
        model = xgb.XGBClassifier(**cfg)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        mlflow.log_param("best_iteration", int(model.best_iteration))
        p_val, p_test = model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]
        res = evaluate_scores(p_val, p_test)
        mlflow.log_metrics(res)
        path = f"/kaggle/working/xgb_v2_final_seed{seed}{SUFFIX}.json"
        model.save_model(path)
        mlflow.log_artifact(path)
    results[seed], scores[seed] = res, (p_val, p_test)
    importances[seed] = dict(zip(feat_names, model.feature_importances_.tolist()))
    print(f"seed {seed}: {model.best_iteration} trees, {_t.time() - t0:.0f}s | val PR-AUC {res['val_pr_auc']:.3f} | test PR-AUC {res['test_pr_auc']:.3f} "
          f"F1 {res['test_f1']:.3f} recall@1%FPR {res['test_recall_at_1pct_fpr']:.3f} precision@1000 {res['test_precision_at_1000']:.3f}")


In [ ]:
import time as _t; print("[cell 6: summary] started", _t.strftime("%H:%M:%S"))
KEYS = ["val_pr_auc", "test_pr_auc", "test_f1", "test_recall_at_1pct_fpr", "test_precision_at_1000", "test_ach_pr_auc", "test_ach_f1"]
summary_rows = {k: (float(np.mean([r[k] for r in results.values()])), float(np.std([r[k] for r in results.values()]))) for k in KEYS}
for k, (m, s) in summary_rows.items():
    print(f"{k:28s} {m:.4f} +- {s:.4f}")

ens_val = np.mean([scores[s][0] for s in SEEDS], axis=0)
ens_test = np.mean([scores[s][1] for s in SEEDS], axis=0)
ens = evaluate_scores(ens_val, ens_test)
print(f"\nensemble of {len(SEEDS)} seeds (average probability): test PR-AUC {ens['test_pr_auc']:.3f}  F1 {ens['test_f1']:.3f}  "
      f"recall@1%FPR {ens['test_recall_at_1pct_fpr']:.3f}  precision@1000 {ens['test_precision_at_1000']:.3f}")

imp = pl.DataFrame({"feature": feat_names, "gain": np.mean([[importances[s][f] for f in feat_names] for s in SEEDS], axis=0)}
                   ).sort("gain", descending=True)
imp_path = f"/kaggle/working/xgb_v2_final_importance{SUFFIX}.csv"
imp.write_csv(imp_path)
pl.Config.set_tbl_rows(15)
print(imp.head(15))

scores_df = pl.DataFrame({
    "txn_id": np.concatenate([val["txn_id"].to_numpy(), test["txn_id"].to_numpy()]),
    "split": ["val"] * len(y_val) + ["test"] * len(y_test),
    "label": np.concatenate([y_val, y_test]).astype(int),
    **{f"xgb_s{s}": np.concatenate(scores[s]) for s in SEEDS},
    "xgb_mean": np.concatenate([ens_val, ens_test]),
})
scores_path = f"/kaggle/working/xgb_scores{SUFFIX}.parquet"
scores_df.write_parquet(scores_path)
summary_path = f"/kaggle/working/xgb_summary{SUFFIX}.json"
with open(summary_path, "w") as fh:
    json.dump({"seeds": list(SEEDS), "per_seed": {str(s): results[s] for s in SEEDS}, "mean_std": summary_rows,
               "ensemble": ens, "n_features": len(feat_names), "features": feat_names}, fh, indent=2)

with mlflow.start_run(run_name=f"xgb_v2_final{SUFFIX}"):
    mlflow.log_param("seeds", len(SEEDS))
    mlflow.log_param("n_features", len(feat_names))
    mlflow.log_metrics({f"{k}_{stat}": v[i] for k, v in summary_rows.items() for i, stat in enumerate(("mean", "std"))})
    mlflow.log_metrics({f"ensemble_{k}": v for k, v in ens.items() if k.startswith(("val_pr", "test_"))})
    for p in (scores_path, summary_path, imp_path):
        mlflow.log_artifact(p)
print("Saved scores, summary and importances (also logged to DagsHub). Download xgb_summary.json, xgb_v2_final_importance.csv "
      "and age_distribution.csv into PaperTrail/reports/.")
